# Advanced Problems with Solutions: Floats — Internal Representation

**Target kernel:** Python 3.13

This notebook contains advanced, solution-driven problems about Python `float` internals.

Topics covered:

- IEEE-754 binary64 layout
- sign, exponent, significand fields
- normal and subnormal numbers
- machine epsilon and ULPs
- exact rational value of floats
- `float.hex()` and `float.fromhex()`
- NaN, infinity, signed zero
- catastrophic cancellation
- stable summation
- float comparison best practices
- decimal-to-binary representation issues


## Setup

We will use only Python standard-library modules.

In [9]:
import math
import struct
import sys
from fractions import Fraction
from decimal import Decimal, getcontext

getcontext().prec = 100

print(sys.version)
print(sys.float_info)

3.13.7 (tags/v3.13.7:bcee1c3, Aug 14 2025, 14:15:11) [MSC v.1944 64 bit (AMD64)]
sys.float_info(max=1.7976931348623157e+308, max_exp=1024, max_10_exp=308, min=2.2250738585072014e-308, min_exp=-1021, min_10_exp=-307, dig=15, mant_dig=53, epsilon=2.220446049250313e-16, radix=2, rounds=1)


# Problem 1 — Decode a Python `float` into IEEE-754 fields

Python `float` is normally implemented as an IEEE-754 binary64 value.

A binary64 float has:

- 1 sign bit
- 11 exponent bits
- 52 stored fraction bits

For normal numbers, the actual significand has an implicit leading `1`, so the precision is effectively 53 bits.

## Task

Write a function `decode_float(x)` that returns:

- sign bit
- biased exponent
- unbiased exponent
- fraction field as an integer
- fraction field as a 52-bit binary string
- category: `zero`, `subnormal`, `normal`, `infinity`, or `nan`
- hexadecimal float representation

Test it on:

```python
0.1, -0.0, 1.0, 0.125, float('inf'), float('-inf'), float('nan'), 5e-324
```

In [10]:
def float_to_uint64(x: float) -> int:
    """Return the raw 64-bit unsigned integer bit pattern of a Python float."""
    return struct.unpack(">Q", struct.pack(">d", x))[0]


def uint64_to_float(bits: int) -> float:
    """Return the Python float represented by a raw 64-bit unsigned integer bit pattern."""
    return struct.unpack(">d", struct.pack(">Q", bits))[0]


def decode_float(x: float) -> dict:
    bits = float_to_uint64(x)

    sign = (bits >> 63) & 1
    exponent_bits = (bits >> 52) & 0x7FF
    fraction_bits = bits & ((1 << 52) - 1)

    if exponent_bits == 0:
        if fraction_bits == 0:
            category = "zero"
            unbiased_exponent = None
        else:
            category = "subnormal"
            unbiased_exponent = -1022
    elif exponent_bits == 0x7FF:
        if fraction_bits == 0:
            category = "infinity"
        else:
            category = "nan"
        unbiased_exponent = None
    else:
        category = "normal"
        unbiased_exponent = exponent_bits - 1023

    return {
        "value": x,
        "raw_bits_hex": f"0x{bits:016x}",
        "raw_bits_binary": f"{bits:064b}",
        "sign": sign,
        "biased_exponent": exponent_bits,
        "unbiased_exponent": unbiased_exponent,
        "fraction_bits_int": fraction_bits,
        "fraction_bits_binary": f"{fraction_bits:052b}",
        "category": category,
        "float_hex": x.hex(),
    }


test_values = [0.1, -0.0, 1.0, 0.125, float("inf"), float("-inf"), float("nan"), 5e-324]

for value in test_values:
    print(decode_float(value))
    print("-" * 80)

{'value': 0.1, 'raw_bits_hex': '0x3fb999999999999a', 'raw_bits_binary': '0011111110111001100110011001100110011001100110011001100110011010', 'sign': 0, 'biased_exponent': 1019, 'unbiased_exponent': -4, 'fraction_bits_int': 2702159776422298, 'fraction_bits_binary': '1001100110011001100110011001100110011001100110011010', 'category': 'normal', 'float_hex': '0x1.999999999999ap-4'}
--------------------------------------------------------------------------------
{'value': -0.0, 'raw_bits_hex': '0x8000000000000000', 'raw_bits_binary': '1000000000000000000000000000000000000000000000000000000000000000', 'sign': 1, 'biased_exponent': 0, 'unbiased_exponent': None, 'fraction_bits_int': 0, 'fraction_bits_binary': '0000000000000000000000000000000000000000000000000000', 'category': 'zero', 'float_hex': '-0x0.0p+0'}
--------------------------------------------------------------------------------
{'value': 1.0, 'raw_bits_hex': '0x3ff0000000000000', 'raw_bits_binary': '00111111111100000000000000000000000

## Solution Notes

The exponent field determines the category:

- exponent bits all zero and fraction zero: signed zero
- exponent bits all zero and fraction nonzero: subnormal
- exponent bits between `1` and `2046`: normal finite number
- exponent bits all ones and fraction zero: infinity
- exponent bits all ones and fraction nonzero: NaN


# Problem 2 — Reconstruct the exact numeric value from raw fields

## Task

Write a function `exact_fraction_from_float(x)` that reconstructs the exact rational value of a finite float using its IEEE-754 fields.

Then compare your result with `x.as_integer_ratio()`.

Test on:

```python
0.1, 0.125, 1.5, -2.75, 5e-324, sys.float_info.min
```

In [11]:
def exact_fraction_from_float(x: float) -> Fraction:
    if math.isnan(x):
        raise ValueError("NaN does not have a rational value")
    if math.isinf(x):
        raise OverflowError("Infinity does not have a finite rational value")

    bits = float_to_uint64(x)
    sign = -1 if ((bits >> 63) & 1) else 1
    exponent_bits = (bits >> 52) & 0x7FF
    fraction_bits = bits & ((1 << 52) - 1)

    if exponent_bits == 0:
        if fraction_bits == 0:
            return Fraction(0, 1)

        # Subnormal:
        # value = (-1)^sign * fraction_bits / 2^52 * 2^-1022
        numerator = sign * fraction_bits
        denominator = 2 ** 1074
        return Fraction(numerator, denominator)

    # Normal:
    # value = (-1)^sign * (1 + fraction_bits / 2^52) * 2^(exponent_bits - 1023)
    significand = (1 << 52) + fraction_bits
    exponent = exponent_bits - 1023 - 52

    if exponent >= 0:
        return Fraction(sign * significand * (2 ** exponent), 1)
    else:
        return Fraction(sign * significand, 2 ** (-exponent))


values = [0.1, 0.125, 1.5, -2.75, 5e-324, sys.float_info.min]

for x in values:
    reconstructed = exact_fraction_from_float(x)
    builtin = Fraction(*x.as_integer_ratio())
    print("x:", x)
    print("reconstructed:", reconstructed)
    print("builtin:      ", builtin)
    print("match:", reconstructed == builtin)
    print()

x: 0.1
reconstructed: 3602879701896397/36028797018963968
builtin:       3602879701896397/36028797018963968
match: True

x: 0.125
reconstructed: 1/8
builtin:       1/8
match: True

x: 1.5
reconstructed: 3/2
builtin:       3/2
match: True

x: -2.75
reconstructed: -11/4
builtin:       -11/4
match: True

x: 5e-324
reconstructed: 1/202402253307310618352495346718917307049556649764142118356901358027430339567995346891960383701437124495187077864316811911389808737385793476867013399940738509921517424276566361364466907742093216341239767678472745068562007483424692698618103355649159556340810056512358769552333414615230502532186327508646006263307707741093494784
builtin:       1/202402253307310618352495346718917307049556649764142118356901358027430339567995346891960383701437124495187077864316811911389808737385793476867013399940738509921517424276566361364466907742093216341239767678472745068562007483424692698618103355649159556340810056512358769552333414615230502532186327508646006263307707741093494784
matc

## Solution Notes

`0.1` is not stored as exactly one tenth. Its exact stored value is a nearby rational number whose denominator is a power of two.

`0.125`, however, is exactly representable because:

```python
0.125 == 1 / 8 == 1 / 2**3
```


# Problem 3 — Show why `0.1 + 0.2 != 0.3`

## Task

Compute the exact rational values of:

```python
0.1
0.2
0.3
0.1 + 0.2
```

Then compute the exact error:

```python
(0.1 + 0.2) - 0.3
```

as a `Fraction`.

In [12]:
a = 0.1
b = 0.2
c = 0.3
s = a + b

for label, value in [("0.1", a), ("0.2", b), ("0.3", c), ("0.1 + 0.2", s)]:
    print(label)
    print("repr:", repr(value))
    print("hex: ", value.hex())
    print("exact:", Fraction(*value.as_integer_ratio()))
    print()

error = Fraction(*s.as_integer_ratio()) - Fraction(*c.as_integer_ratio())

print("0.1 + 0.2 == 0.3:", s == c)
print("exact error:", error)
print("error as float:", float(error))
print("error in scientific notation:", format(float(error), ".25e"))

0.1
repr: 0.1
hex:  0x1.999999999999ap-4
exact: 3602879701896397/36028797018963968

0.2
repr: 0.2
hex:  0x1.999999999999ap-3
exact: 3602879701896397/18014398509481984

0.3
repr: 0.3
hex:  0x1.3333333333333p-2
exact: 5404319552844595/18014398509481984

0.1 + 0.2
repr: 0.30000000000000004
hex:  0x1.3333333333334p-2
exact: 1351079888211149/4503599627370496

0.1 + 0.2 == 0.3: False
exact error: 1/18014398509481984
error as float: 5.551115123125783e-17
error in scientific notation: 5.5511151231257827021181583e-17


## Solution Notes

The expression `0.1 + 0.2` produces the nearest representable binary64 result to the real-number sum of the two already-rounded binary64 inputs.

The final result is slightly greater than the float stored for `0.3`, so equality fails.

Best practice: do not compare most computed floats using direct equality. Use tolerances or exact types when appropriate.

# Problem 4 — Implement your own `ulp(x)` helper

The ULP of a finite float is the distance between that float and the next larger representable float.

Python provides `math.ulp(x)`, but here you will implement a simplified positive finite version yourself.

## Task

Write `my_ulp(x)` for positive finite floats.

Compare it with `math.ulp(x)` for:

```python
0.0, 5e-324, 1e-320, 1.0, 1.5, 2.0, 1e16
```

In [13]:
def next_float_up(x: float) -> float:
    if math.isnan(x):
        return x
    if x == math.inf:
        return x
    if x == 0.0:
        return uint64_to_float(1)

    bits = float_to_uint64(x)

    if x > 0:
        bits += 1
    else:
        bits -= 1

    return uint64_to_float(bits)


def my_ulp(x: float) -> float:
    if math.isnan(x):
        return math.nan
    if math.isinf(x):
        return math.inf
    if x < 0:
        x = -x
    return next_float_up(x) - x


values = [0.0, 5e-324, 1e-320, 1.0, 1.5, 2.0, 1e16]

for x in values:
    print("x:", x)
    print("my_ulp:  ", my_ulp(x))
    print("math.ulp:", math.ulp(x))
    print("match:   ", my_ulp(x) == math.ulp(x))
    print()

x: 0.0
my_ulp:   5e-324
math.ulp: 5e-324
match:    True

x: 5e-324
my_ulp:   5e-324
math.ulp: 5e-324
match:    True

x: 1e-320
my_ulp:   5e-324
math.ulp: 5e-324
match:    True

x: 1.0
my_ulp:   2.220446049250313e-16
math.ulp: 2.220446049250313e-16
match:    True

x: 1.5
my_ulp:   2.220446049250313e-16
math.ulp: 2.220446049250313e-16
match:    True

x: 2.0
my_ulp:   4.440892098500626e-16
math.ulp: 4.440892098500626e-16
match:    True

x: 1e+16
my_ulp:   2.0
math.ulp: 2.0
match:    True



## Solution Notes

Around `1.0`, the spacing between adjacent floats is `2**-52`.

Around `2.0`, the spacing doubles to `2**-51`.

Float spacing is not uniform across the number line. It grows as the exponent grows.

# Problem 5 — Machine epsilon versus ULP

Machine epsilon is often described as the smallest positive number `eps` such that:

```python
1.0 + eps != 1.0
```

For Python binary64 floats, this value is `2**-52`.

## Task

Find machine epsilon experimentally by repeatedly halving a number.

Then compare:

```python
eps
sys.float_info.epsilon
math.ulp(1.0)
math.ulp(0.5)
math.ulp(2.0)
```

In [14]:
def find_machine_epsilon() -> float:
    eps = 1.0
    while 1.0 + eps / 2.0 != 1.0:
        eps /= 2.0
    return eps


eps = find_machine_epsilon()

print("experimental epsilon:", eps)
print("sys.float_info.epsilon:", sys.float_info.epsilon)
print("math.ulp(1.0):", math.ulp(1.0))
print("math.ulp(0.5):", math.ulp(0.5))
print("math.ulp(2.0):", math.ulp(2.0))

assert eps == sys.float_info.epsilon
assert eps == math.ulp(1.0)

experimental epsilon: 2.220446049250313e-16
sys.float_info.epsilon: 2.220446049250313e-16
math.ulp(1.0): 2.220446049250313e-16
math.ulp(0.5): 1.1102230246251565e-16
math.ulp(2.0): 4.440892098500626e-16


## Solution Notes

Machine epsilon is specifically tied to spacing at `1.0`.

It is not the smallest positive float. The smallest positive subnormal float is much smaller:

```python
5e-324
```

# Problem 6 — Investigate signed zero

IEEE-754 has both positive zero and negative zero.

In Python:

```python
0.0 == -0.0
```

is true, but the two values have different bit patterns.

## Task

Show that `0.0` and `-0.0`:

- compare equal
- have the same hash
- have different raw bits
- can be distinguished with `math.copysign`
- have different hexadecimal representations

In [15]:
positive_zero = 0.0
negative_zero = -0.0

print("0.0 == -0.0:", positive_zero == negative_zero)
print("hash(0.0):", hash(positive_zero))
print("hash(-0.0):", hash(negative_zero))
print("bits 0.0: ", f"0x{float_to_uint64(positive_zero):016x}")
print("bits -0.0:", f"0x{float_to_uint64(negative_zero):016x}")
print("copysign for 0.0: ", math.copysign(1.0, positive_zero))
print("copysign for -0.0:", math.copysign(1.0, negative_zero))
print("0.0.hex(): ", positive_zero.hex())
print("-0.0.hex():", negative_zero.hex())

0.0 == -0.0: True
hash(0.0): 0
hash(-0.0): 0
bits 0.0:  0x0000000000000000
bits -0.0: 0x8000000000000000
copysign for 0.0:  1.0
copysign for -0.0: -1.0
0.0.hex():  0x0.0p+0
-0.0.hex(): -0x0.0p+0


## Solution Notes

Signed zero matters in some numerical algorithms, especially where branch cuts, limits, complex numbers, or directional behavior are involved.

For ordinary arithmetic, `0.0` and `-0.0` usually behave as equal values.

# Problem 7 — Explore NaN behavior

`NaN` means Not a Number.

A key property of NaN is that it does not compare equal to anything, including itself.

## Task

Create a NaN and show:

- `nan == nan` is false
- `nan != nan` is true
- ordering comparisons are false
- `math.isnan` detects it
- NaNs can have different bit patterns
- direct equality is not a valid NaN test

In [16]:
nan1 = float("nan")
nan2 = uint64_to_float(0x7ff8000000000001)
nan3 = uint64_to_float(0x7ff8000000000002)

print("nan1 == nan1:", nan1 == nan1)
print("nan1 != nan1:", nan1 != nan1)
print("nan1 < 0:", nan1 < 0)
print("nan1 > 0:", nan1 > 0)
print("nan1 <= 0:", nan1 <= 0)
print("nan1 >= 0:", nan1 >= 0)
print("math.isnan(nan1):", math.isnan(nan1))
print()

for label, value in [("nan1", nan1), ("nan2", nan2), ("nan3", nan3)]:
    print(label, value.hex(), f"0x{float_to_uint64(value):016x}", math.isnan(value))

nan1 == nan1: False
nan1 != nan1: True
nan1 < 0: False
nan1 > 0: False
nan1 <= 0: False
nan1 >= 0: False
math.isnan(nan1): True

nan1 nan 0x7ff8000000000000 True
nan2 nan 0x7ff8000000000001 True
nan3 nan 0x7ff8000000000002 True


## Solution Notes

Use:

```python
math.isnan(x)
```

not:

```python
x == float('nan')
```

NaN is intentionally not equal to itself.

# Problem 8 — Find the largest finite float and the overflow boundary

## Task

Use `sys.float_info.max` and `math.nextafter` to investigate the largest finite float.

Show:

- the largest finite float
- its hex representation
- the next float after it toward positive infinity
- what happens when multiplying it by `2.0`
- the previous float before positive infinity

In [17]:
largest = sys.float_info.max

print("largest finite float:", largest)
print("hex:", largest.hex())
print("bits:", f"0x{float_to_uint64(largest):016x}")
print("nextafter(largest, inf):", math.nextafter(largest, math.inf))
print("largest * 2.0:", largest * 2.0)
print("nextafter(inf, 0.0):", math.nextafter(math.inf, 0.0))
print("same as largest:", math.nextafter(math.inf, 0.0) == largest)

largest finite float: 1.7976931348623157e+308
hex: 0x1.fffffffffffffp+1023
bits: 0x7fefffffffffffff
nextafter(largest, inf): inf
largest * 2.0: inf
nextafter(inf, 0.0): 1.7976931348623157e+308
same as largest: True


## Solution Notes

The largest finite binary64 float has exponent bits `2046` and all fraction bits set.

The next representable value above it is positive infinity.

# Problem 9 — Subnormal numbers and gradual underflow

Subnormal numbers fill the gap between zero and the smallest positive normal float.

## Task

Show:

- smallest positive subnormal float
- smallest positive normal float
- the number of ULP steps from zero to the smallest normal float
- that subnormal spacing is constant
- that repeated halving of the smallest normal eventually reaches zero

In [18]:
smallest_subnormal = math.ulp(0.0)
smallest_normal = sys.float_info.min

print("smallest positive subnormal:", smallest_subnormal)
print("hex:", smallest_subnormal.hex())
print("bits:", f"0x{float_to_uint64(smallest_subnormal):016x}")
print()

print("smallest positive normal:", smallest_normal)
print("hex:", smallest_normal.hex())
print("bits:", f"0x{float_to_uint64(smallest_normal):016x}")
print()

steps = float_to_uint64(smallest_normal) - float_to_uint64(0.0)
print("ULP steps from +0.0 to smallest normal:", steps)
print("expected 2**52:", 2**52)
print("match:", steps == 2**52)
print()

print("ulp(0.0):", math.ulp(0.0))
print("ulp(smallest_subnormal):", math.ulp(smallest_subnormal))
print("ulp(smallest_normal / 2):", math.ulp(smallest_normal / 2))
print()

x = smallest_normal
count = 0
while x != 0.0:
    x /= 2.0
    count += 1

print("number of halvings from smallest normal to zero:", count)

smallest positive subnormal: 5e-324
hex: 0x0.0000000000001p-1022
bits: 0x0000000000000001

smallest positive normal: 2.2250738585072014e-308
hex: 0x1.0000000000000p-1022
bits: 0x0010000000000000

ULP steps from +0.0 to smallest normal: 4503599627370496
expected 2**52: 4503599627370496
match: True

ulp(0.0): 5e-324
ulp(smallest_subnormal): 5e-324
ulp(smallest_normal / 2): 5e-324

number of halvings from smallest normal to zero: 53


## Solution Notes

Subnormal numbers sacrifice significand precision to provide gradual underflow.

Without subnormals, values smaller than the smallest normal would immediately underflow to zero.

# Problem 10 — Use `float.hex()` as a lossless representation

Decimal display is often rounded for readability.

`float.hex()` gives a precise base-16 representation of the stored binary float.

## Task

For several floats:

- print `repr(x)`
- print `x.hex()`
- reconstruct with `float.fromhex(x.hex())`
- verify equality

Use:

```python
0.1, math.pi, 1.1 + 2.2, 5e-324, sys.float_info.max
```

In [19]:
values = [0.1, math.pi, 1.1 + 2.2, 5e-324, sys.float_info.max]

for x in values:
    h = x.hex()
    y = float.fromhex(h)
    print("repr:", repr(x))
    print("hex: ", h)
    print("round trip equal:", x == y)
    print("bits equal:", float_to_uint64(x) == float_to_uint64(y))
    print()

repr: 0.1
hex:  0x1.999999999999ap-4
round trip equal: True
bits equal: True

repr: 3.141592653589793
hex:  0x1.921fb54442d18p+1
round trip equal: True
bits equal: True

repr: 3.3000000000000003
hex:  0x1.a666666666667p+1
round trip equal: True
bits equal: True

repr: 5e-324
hex:  0x0.0000000000001p-1022
round trip equal: True
bits equal: True

repr: 1.7976931348623157e+308
hex:  0x1.fffffffffffffp+1023
round trip equal: True
bits equal: True



## Solution Notes

`float.hex()` is excellent for debugging and serialization when you want exact round-tripping of float values.

# Problem 11 — Demonstrate catastrophic cancellation

Catastrophic cancellation occurs when subtracting nearly equal floating-point numbers.

## Task

Compare the following two mathematically equivalent expressions:

```python
sqrt(x + 1) - sqrt(x)
1 / (sqrt(x + 1) + sqrt(x))
```

for large `x`.

Use:

```python
x = 10**16
```

In [20]:
x = 10**16

unstable = math.sqrt(x + 1) - math.sqrt(x)
stable = 1 / (math.sqrt(x + 1) + math.sqrt(x))

print("unstable:", unstable)
print("stable:  ", stable)
print("absolute difference:", abs(unstable - stable))
print("relative difference:", abs(unstable - stable) / stable)
print()

print("sqrt(x + 1):", math.sqrt(x + 1))
print("sqrt(x):    ", math.sqrt(x))
print("equal sqrt results:", math.sqrt(x + 1) == math.sqrt(x))

unstable: 0.0
stable:   5e-09
absolute difference: 5e-09
relative difference: 1.0

sqrt(x + 1): 100000000.0
sqrt(x):     100000000.0
equal sqrt results: True


## Solution Notes

The unstable expression subtracts two nearly equal rounded quantities.

The stable expression avoids that subtraction by rationalizing the numerator:

```text
sqrt(x + 1) - sqrt(x)
= ((sqrt(x + 1) - sqrt(x)) * (sqrt(x + 1) + sqrt(x))) / (sqrt(x + 1) + sqrt(x))
= 1 / (sqrt(x + 1) + sqrt(x))
```

# Problem 12 — Naive summation versus `math.fsum`

Floating-point addition is not associative.

That means:

```python
(a + b) + c
```

may differ from:

```python
a + (b + c)
```

## Task

Use this list:

```python
data = [1e16, 1.0, -1e16]
```

Compare:

- `sum(data)`
- `math.fsum(data)`
- different addition orders

Then test a larger list containing many small terms.

In [21]:
data = [1e16, 1.0, -1e16]

print("data:", data)
print("sum(data):", sum(data))
print("math.fsum(data):", math.fsum(data))
print()

a, b, c = data
print("(a + b) + c:", (a + b) + c)
print("a + (b + c):", a + (b + c))
print()

many = [1.0] + [1e-16] * 1_000_000
print("Expected approximately:", 1.0 + 1_000_000 * 1e-16)
print("sum(many):", sum(many))
print("math.fsum(many):", math.fsum(many))

data: [1e+16, 1.0, -1e+16]
sum(data): 1.0
math.fsum(data): 1.0

(a + b) + c: 0.0
a + (b + c): 0.0

Expected approximately: 1.0000000001
sum(many): 1.0000000001
math.fsum(many): 1.0000000001


## Solution Notes

`math.fsum` tracks partial sums more carefully and often gives a more accurate result than the built-in `sum` for floating-point data.

Best practice:

- use `math.fsum` for high-accuracy summation of floats,
- avoid assuming floating-point addition is associative,
- be careful when summing values with very different magnitudes.

# Problem 13 — Implement a robust float comparison helper

Direct equality is often inappropriate for computed floats.

Python provides:

```python
math.isclose(a, b, rel_tol=..., abs_tol=...)
```

## Task

Implement your own simplified `is_close(a, b, rel_tol, abs_tol)` using this rule:

```python
abs(a - b) <= max(rel_tol * max(abs(a), abs(b)), abs_tol)
```

Compare it with `math.isclose`.

In [22]:
def is_close(a: float, b: float, *, rel_tol: float = 1e-9, abs_tol: float = 0.0) -> bool:
    if rel_tol < 0 or abs_tol < 0:
        raise ValueError("tolerances must be non-negative")

    if a == b:
        return True

    if math.isinf(a) or math.isinf(b):
        return False

    if math.isnan(a) or math.isnan(b):
        return False

    return abs(a - b) <= max(rel_tol * max(abs(a), abs(b)), abs_tol)


pairs = [
    (0.1 + 0.2, 0.3),
    (1.000000001, 1.0),
    (1e100 + 1e90, 1e100),
    (1e-12, 0.0),
    (math.inf, math.inf),
    (math.inf, 1.0),
    (math.nan, math.nan),
]

for a, b in pairs:
    print("a:", a, "b:", b)
    print("==:", a == b)
    print("is_close:", is_close(a, b, rel_tol=1e-9, abs_tol=1e-12))
    print("math.isclose:", math.isclose(a, b, rel_tol=1e-9, abs_tol=1e-12))
    print()

a: 0.30000000000000004 b: 0.3
==: False
is_close: True
math.isclose: True

a: 1.000000001 b: 1.0
==: False
is_close: False
math.isclose: False

a: 1.0000000001e+100 b: 1e+100
==: False
is_close: True
math.isclose: True

a: 1e-12 b: 0.0
==: False
is_close: True
math.isclose: True

a: inf b: inf
==: True
is_close: True
math.isclose: True

a: inf b: 1.0
==: False
is_close: False
math.isclose: False

a: nan b: nan
==: False
is_close: False
math.isclose: False



## Solution Notes

Relative tolerance works well for large-magnitude values.

Absolute tolerance is essential when comparing values near zero.

Best practice:

```python
math.isclose(a, b, rel_tol=..., abs_tol=...)
```

Use tolerances that make sense for your problem domain.

# Problem 14 — Find the next representable floats around a value

## Task

For a given finite float `x`, write a function `neighbors(x)` that returns:

- previous representable float
- `x`
- next representable float
- distance to previous
- distance to next

Test on:

```python
0.0, -0.0, 1.0, 2.0, -1.0, 1e300, 5e-324
```

In [23]:
def neighbors(x: float) -> dict:
    previous_value = math.nextafter(x, -math.inf)
    next_value = math.nextafter(x, math.inf)

    return {
        "previous": previous_value,
        "x": x,
        "next": next_value,
        "distance_to_previous": x - previous_value,
        "distance_to_next": next_value - x,
        "previous_hex": previous_value.hex(),
        "x_hex": x.hex(),
        "next_hex": next_value.hex(),
    }


for x in [0.0, -0.0, 1.0, 2.0, -1.0, 1e300, 5e-324]:
    print("x =", x)
    for key, value in neighbors(x).items():
        print(f"  {key}: {value}")
    print()

x = 0.0
  previous: -5e-324
  x: 0.0
  next: 5e-324
  distance_to_previous: 5e-324
  distance_to_next: 5e-324
  previous_hex: -0x0.0000000000001p-1022
  x_hex: 0x0.0p+0
  next_hex: 0x0.0000000000001p-1022

x = -0.0
  previous: -5e-324
  x: -0.0
  next: 5e-324
  distance_to_previous: 5e-324
  distance_to_next: 5e-324
  previous_hex: -0x0.0000000000001p-1022
  x_hex: -0x0.0p+0
  next_hex: 0x0.0000000000001p-1022

x = 1.0
  previous: 0.9999999999999999
  x: 1.0
  next: 1.0000000000000002
  distance_to_previous: 1.1102230246251565e-16
  distance_to_next: 2.220446049250313e-16
  previous_hex: 0x1.fffffffffffffp-1
  x_hex: 0x1.0000000000000p+0
  next_hex: 0x1.0000000000001p+0

x = 2.0
  previous: 1.9999999999999998
  x: 2.0
  next: 2.0000000000000004
  distance_to_previous: 2.220446049250313e-16
  distance_to_next: 4.440892098500626e-16
  previous_hex: 0x1.fffffffffffffp+0
  x_hex: 0x1.0000000000000p+1
  next_hex: 0x1.0000000000001p+1

x = -1.0
  previous: -1.0000000000000002
  x: -1.0
  nex

## Solution Notes

`math.nextafter(x, y)` returns the next representable float after `x` in the direction of `y`.

This is useful for testing numerical boundary cases.

# Problem 15 — Determine whether a rational number is exactly representable as a float

A rational number is exactly representable as a finite binary fraction if, after reducing the fraction, its denominator is a power of two.

However, binary64 also has range and precision limits.

## Task

Write a function `has_terminating_binary_expansion(frac)` that checks whether a `Fraction` has a terminating binary expansion.

Test it on:

```python
Fraction(1, 10)
Fraction(1, 8)
Fraction(3, 16)
Fraction(22, 7)
Fraction(1, 5)
Fraction(7, 64)
```

In [24]:
def is_power_of_two(n: int) -> bool:
    return n > 0 and (n & (n - 1)) == 0


def has_terminating_binary_expansion(frac: Fraction) -> bool:
    frac = Fraction(frac)
    return is_power_of_two(frac.denominator)


fractions = [
    Fraction(1, 10),
    Fraction(1, 8),
    Fraction(3, 16),
    Fraction(22, 7),
    Fraction(1, 5),
    Fraction(7, 64),
]

for frac in fractions:
    print(frac, "terminating binary:", has_terminating_binary_expansion(frac), "float:", float(frac))

1/10 terminating binary: False float: 0.1
1/8 terminating binary: True float: 0.125
3/16 terminating binary: True float: 0.1875
22/7 terminating binary: False float: 3.142857142857143
1/5 terminating binary: False float: 0.2
7/64 terminating binary: True float: 0.109375


## Solution Notes

`1/10` has a terminating decimal expansion but not a terminating binary expansion.

`1/8` has both a terminating decimal expansion and a terminating binary expansion.

That is why `0.125` is represented exactly but `0.1` is not.

# Problem 16 — Inspect rounding when converting decimal strings to floats

When Python evaluates:

```python
float('0.1')
```

it returns the nearest representable binary64 float to the exact decimal value `0.1`.

## Task

Find the float immediately below and above `0.1`.

Then compare their exact distances to the mathematical value `1/10`.

In [25]:
target = Fraction(1, 10)
x = float("0.1")
below = math.nextafter(x, -math.inf)
above = math.nextafter(x, math.inf)

candidates = [("below", below), ("x", x), ("above", above)]

for label, value in candidates:
    exact = Fraction(*value.as_integer_ratio())
    distance = abs(exact - target)
    print(label)
    print("  repr:", repr(value))
    print("  hex: ", value.hex())
    print("  exact:", exact)
    print("  distance from 1/10:", distance)
    print("  distance as decimal:", Decimal(distance.numerator) / Decimal(distance.denominator))
    print()

below
  repr: 0.09999999999999999
  hex:  0x1.9999999999999p-4
  exact: 7205759403792793/72057594037927936
  distance from 1/10: 3/360287970189639680
  distance as decimal: 8.32667268468867405317723751068115234375E-18

x
  repr: 0.1
  hex:  0x1.999999999999ap-4
  exact: 3602879701896397/36028797018963968
  distance from 1/10: 1/180143985094819840
  distance as decimal: 5.5511151231257827021181583404541015625E-18

above
  repr: 0.10000000000000002
  hex:  0x1.999999999999bp-4
  exact: 7205759403792795/72057594037927936
  distance from 1/10: 7/360287970189639680
  distance as decimal: 1.942890293094023945741355419158935546875E-17



## Solution Notes

The stored value for `0.1` is the representable binary64 float closest to the exact rational value `1/10`.

The displayed decimal `0.1` is not the full exact value. Python chooses a short decimal representation that round-trips to the same binary float.

# Problem 17 — Build an ULP-distance function

For many numerical tests, it is useful to measure how many representable float steps separate two floats.

## Task

Write `ulp_distance(a, b)` for finite floats.

The function should map float bit patterns to an ordered integer space so that adjacent floats differ by 1.

Test it on nearby values around `1.0`, `0.0`, and negative numbers.

In [26]:
def ordered_float_int(x: float) -> int:
    """Map a float to an integer that preserves numerical ordering."""
    bits = float_to_uint64(x)

    if bits & (1 << 63):
        # Negative numbers: reverse ordering.
        return ~bits & 0xFFFFFFFFFFFFFFFF
    else:
        # Positive numbers: shift above negative range.
        return bits | (1 << 63)


def ulp_distance(a: float, b: float) -> int:
    if not math.isfinite(a) or not math.isfinite(b):
        raise ValueError("ulp_distance requires finite floats")
    return abs(ordered_float_int(a) - ordered_float_int(b))


pairs = [
    (1.0, math.nextafter(1.0, math.inf)),
    (1.0, math.nextafter(math.nextafter(1.0, math.inf), math.inf)),
    (0.0, 5e-324),
    (-0.0, 0.0),
    (-1.0, math.nextafter(-1.0, -math.inf)),
    (-1.0, math.nextafter(-1.0, math.inf)),
]

for a, b in pairs:
    print("a:", a, a.hex())
    print("b:", b, b.hex())
    print("ulp distance:", ulp_distance(a, b))
    print()

a: 1.0 0x1.0000000000000p+0
b: 1.0000000000000002 0x1.0000000000001p+0
ulp distance: 1

a: 1.0 0x1.0000000000000p+0
b: 1.0000000000000004 0x1.0000000000002p+0
ulp distance: 2

a: 0.0 0x0.0p+0
b: 5e-324 0x0.0000000000001p-1022
ulp distance: 1

a: -0.0 -0x0.0p+0
b: 0.0 0x0.0p+0
ulp distance: 1

a: -1.0 -0x1.0000000000000p+0
b: -1.0000000000000002 -0x1.0000000000001p+0
ulp distance: 1

a: -1.0 -0x1.0000000000000p+0
b: -0.9999999999999999 -0x1.fffffffffffffp-1
ulp distance: 1



## Solution Notes

Raw unsigned integer bit patterns do not directly sort like floats because of the sign bit.

The helper `ordered_float_int` transforms bit patterns so that numerical order is preserved.

# Problem 18 — Compare `float`, `Decimal`, and `Fraction`

Different numeric types make different trade-offs.

## Task

Compute `0.1 + 0.2` using:

- float
- Decimal from strings
- Fraction from strings

Compare each result to `0.3` represented in the same type.

In [27]:
float_result = 0.1 + 0.2
float_target = 0.3

decimal_result = Decimal("0.1") + Decimal("0.2")
decimal_target = Decimal("0.3")

fraction_result = Fraction("0.1") + Fraction("0.2")
fraction_target = Fraction("0.3")

print("float result:   ", float_result, "equal to 0.3?", float_result == float_target)
print("Decimal result: ", decimal_result, "equal to 0.3?", decimal_result == decimal_target)
print("Fraction result:", fraction_result, "equal to 3/10?", fraction_result == fraction_target)
print()

print("Exact float result as Fraction:", Fraction(*float_result.as_integer_ratio()))
print("Exact float target as Fraction:", Fraction(*float_target.as_integer_ratio()))

float result:    0.30000000000000004 equal to 0.3? False
Decimal result:  0.3 equal to 0.3? True
Fraction result: 3/10 equal to 3/10? True

Exact float result as Fraction: 1351079888211149/4503599627370496
Exact float target as Fraction: 5404319552844595/18014398509481984


## Solution Notes

Use the right numeric type for the job:

- `float`: fast approximate binary floating point
- `Decimal`: base-10 arithmetic with configurable precision, useful for many financial-style calculations
- `Fraction`: exact rational arithmetic, useful for symbolic or exact reasoning but can be slower and produce large numerators/denominators

# Problem 19 — Rounding half to even

Python's `round` uses round-half-to-even for exact halfway cases.

## Task

Investigate:

```python
round(0.5)
round(1.5)
round(2.5)
round(3.5)
```

Then investigate why:

```python
round(2.675, 2)
```

may surprise people.

In [28]:
for x in [0.5, 1.5, 2.5, 3.5, 4.5, 5.5]:
    print(f"round({x}) =", round(x))

print()

x = 2.675
print("x:", x)
print("repr(x):", repr(x))
print("x.hex():", x.hex())
print("format(x, '.30f'):", format(x, ".30f"))
print("round(x, 2):", round(x, 2))
print("exact stored value:", Fraction(*x.as_integer_ratio()))
print("stored value as Decimal:", Decimal(x.as_integer_ratio()[0]) / Decimal(x.as_integer_ratio()[1]))

round(0.5) = 0
round(1.5) = 2
round(2.5) = 2
round(3.5) = 4
round(4.5) = 4
round(5.5) = 6

x: 2.675
repr(x): 2.675
x.hex(): 0x1.5666666666666p+1
format(x, '.30f'): 2.674999999999999822364316059975
round(x, 2): 2.67
exact stored value: 3011782250804019/1125899906842624
stored value as Decimal: 2.67499999999999982236431605997495353221893310546875


## Solution Notes

`round(2.675, 2)` surprises people because the stored binary float is not exactly the decimal value `2.675`.

The rounding operation is applied to the stored binary approximation, not to an exact decimal literal.

# Problem 20 — Best-practice diagnostic function for floats

## Task

Create a function `float_report(x)` that prints a useful diagnostic report for any Python float.

The report should include:

- `repr(x)`
- `str(x)`
- `x.hex()`
- raw bits
- category
- exact rational value if finite
- previous and next representable floats if finite
- ULP if finite

Test it on:

```python
0.1, 1.0, -0.0, 5e-324, sys.float_info.max, math.inf, math.nan
```

In [29]:
def float_report(x: float) -> None:
    info = decode_float(x)

    print("value repr:       ", repr(x))
    print("value str:        ", str(x))
    print("hex:              ", x.hex())
    print("raw bits:         ", info["raw_bits_hex"])
    print("category:         ", info["category"])
    print("sign:             ", info["sign"])
    print("biased exponent:  ", info["biased_exponent"])
    print("unbiased exponent:", info["unbiased_exponent"])
    print("fraction bits:    ", info["fraction_bits_binary"])

    if math.isfinite(x):
        exact = Fraction(*x.as_integer_ratio())
        print("exact fraction:   ", exact)
        print("exact decimal:    ", Decimal(exact.numerator) / Decimal(exact.denominator))
        print("previous float:   ", math.nextafter(x, -math.inf))
        print("next float:       ", math.nextafter(x, math.inf))
        print("ulp:              ", math.ulp(x))
    else:
        print("exact fraction:    not finite")
        print("previous float:    not applicable")
        print("next float:        not applicable")
        print("ulp:               not finite")


for x in [0.1, 1.0, -0.0, 5e-324, sys.float_info.max, math.inf, math.nan]:
    print("=" * 90)
    float_report(x)

value repr:        0.1
value str:         0.1
hex:               0x1.999999999999ap-4
raw bits:          0x3fb999999999999a
category:          normal
sign:              0
biased exponent:   1019
unbiased exponent: -4
fraction bits:     1001100110011001100110011001100110011001100110011010
exact fraction:    3602879701896397/36028797018963968
exact decimal:     0.1000000000000000055511151231257827021181583404541015625
previous float:    0.09999999999999999
next float:        0.10000000000000002
ulp:               1.3877787807814457e-17
value repr:        1.0
value str:         1.0
hex:               0x1.0000000000000p+0
raw bits:          0x3ff0000000000000
category:          normal
sign:              0
biased exponent:   1023
unbiased exponent: 0
fraction bits:     0000000000000000000000000000000000000000000000000000
exact fraction:    1
exact decimal:     1
previous float:    0.9999999999999999
next float:        1.0000000000000002
ulp:               2.220446049250313e-16
value repr:  

# Final Best Practices Summary

## 1. Remember that Python `float` is approximate

Most decimal fractions are not exactly representable as binary floating-point numbers.

Examples:

```python
0.1
0.2
0.3
2.675
```

are stored as nearby binary64 approximations.

## 2. Do not use direct equality for most computed floats

Prefer:

```python
math.isclose(a, b, rel_tol=..., abs_tol=...)
```

## 3. Use `math.fsum` for accurate summation

Prefer:

```python
math.fsum(values)
```

over:

```python
sum(values)
```

when numerical accuracy matters.

## 4. Use `float.hex()` for exact float debugging

This round-trips exactly:

```python
x == float.fromhex(x.hex())
```

## 5. Use `Decimal` or `Fraction` when appropriate

Use:

- `Decimal` for decimal-oriented arithmetic
- `Fraction` for exact rational arithmetic
- `float` for fast approximate real-number arithmetic

## 6. Know the special values

Floats include:

- positive infinity
- negative infinity
- NaN
- positive zero
- negative zero
- subnormal numbers

Use `math.isnan`, `math.isinf`, and `math.isfinite` for classification.